In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import trimesh
import fvdb
import argparse
from pprint import pprint
import tqdm
import h5py
from skimage import measure
from meshplot import plot
from scipy.spatial.distance import cdist
import joblib
import numpy as np
from scipy.spatial import KDTree
import igl
import fvdb.nn as fvnn

# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
import eval
import utils
import models
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 
from utils import mesh_tools as mt

In [2]:
gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
gt =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt'

os.listdir(gt_large)
# read hdf5 file
import h5py
h5_file = h5py.File(os.path.join(gt_large, '00000020.hdf5'), 'r')
h5_file_gt = h5py.File(os.path.join(gt, '00000020.hdf5'), 'r')
sdf_128_gt = h5_file_gt['128_sdf'][:]
# print all keys
print(list(h5_file.keys()))
# read sdf
sdf_32 = h5_file['32_sdf'][:]
sdf_64 = h5_file['64_sdf'][:]
sdf_128 = h5_file['128_sdf'][:]
sdf_256 = h5_file['256_sdf'][:]
print(sdf_32.shape)
print(sdf_64.shape)
print(sdf_128.shape)
print(sdf_256.shape)
(abs(sdf_128_gt - sdf_128)>0.001).sum()

['128_sdf', '256_sdf', '32_sdf', '64_sdf']
(33, 33, 33)
(65, 65, 65)
(129, 129, 129)
(257, 257, 257)


np.int64(0)

In [17]:
mask_32 = mt.make_mask_close(sdf_32, 32)
filter_sdf_32 = sdf_32*mask_32
print(filter_sdf_32.shape)
print(f'total number of true values in mask: {np.sum(mask_32)}')
mt.plotSlice(filter_sdf_32, 0.1)

(33, 33, 33)
total number of true values in mask: 5171


interactive(children=(IntSlider(value=16, description='s', max=32), Output()), _dom_classes=('widget-interact'…

<function mesh_tools.plotSlice.<locals>.<lambda>(s)>

In [23]:
mask_128 = mt.make_mask_close(sdf_128, (128*2)+1)
filter_sdf_128 = sdf_128*mask_128
print(filter_sdf_128.shape)
print(f'total number of true values in mask: {np.sum(mask_128)}')
mt.plotSlice(filter_sdf_128, 0.05)
mt.plotSlice(sdf_128, 0.05)
v, f, _, _ = measure.marching_cubes(filter_sdf_128)
plot(v, f)

(129, 129, 129)
total number of true values in mask: 54012


interactive(children=(IntSlider(value=64, description='s', max=128), Output()), _dom_classes=('widget-interact…

interactive(children=(IntSlider(value=64, description='s', max=128), Output()), _dom_classes=('widget-interact…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(63.998571…

In [5]:
filter_sdf_128[~mask_128]= 100
v, f, _, _ = measure.marching_cubes(filter_sdf_128, level=0.0)
plot(v,f)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(63.998571…

In [15]:
mask_256 = mt.make_mask_close(sdf_256, 256*2)
filter_sdf_256 = sdf_256*mask_256
print(filter_sdf_256.shape)
print(f'total number of true values in mask: {np.sum(mask_256)}')
mt.plotSlice(filter_sdf_256, 0.005)

NameError: name 'sdf_256' is not defined

In [7]:
filter_sdf_256[~mask_256]= 100
v, f, _, _ = measure.marching_cubes(filter_sdf_256, level=0.0)
plot(v,f)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(127.99752…

In [8]:
test_predictions_dir = '/data/workspaces/spanwar/results/ssu/test_predictions'
os.listdir(test_predictions_dir)

['SSU_PONQ_DATA_UPSAMPLER_Unet',
 '26_rerun_24_SSU_PONQ_DATA_UPSAMPLER_FM_trilinear_Unet_v2',
 '27_Unet_Upsampler_Manual_NMC_data',
 '28_Unet_Upsampler_Manual_NMC_data_l1',
 '29_Unet_Upsampler_Manual_NMC_data_l1',
 '30_Unet_Upsampler_Manual_NMC_data_l1',
 '31_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '32_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '33_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '34_Simple_CNN_Upsampler_Manual_NMC_data_huber_loss',
 '35_rerun_33_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '36_rerun_35_Simple_CNN_Upsampler_Manual_NMC_data_l1',
 '37_residual_CNN_Upsampler_Manual_NMC_data_l1',
 '38_rerun_37_residual_CNN_Upsampler_Manual_NMC_data_l1',
 '39_residual_CNN_Upsampler_Manual_NMC_data_l1_&_sign_loss',
 '40_rerun_31_CNN_Upsampler_Manual_NMC_data_l1_&_sign_loss',
 '41_same_40_eval_level_3',
 '42_rerun_40_CNN_vanilla_65_mask_threshold_l1_sign_loss',
 '44_CNN_vanilla_FM',
 '45_rerun_44_more_epochs_CNN_vanilla_FM',
 '46_rerun_45_with_Unet',
 '47_test_44_test_optimization

In [9]:
sdf_128.shape

(129, 129, 129)

In [10]:
def eval_sdf(model_name, sub_dir=None):
    gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    test_predictions_dir = '/data/workspaces/spanwar/results/ssu/test_predictions'
    if sub_dir is not None:
        model_name = os.path.join(model_name, sub_dir)
    model_predictions_dir = os.path.join(test_predictions_dir, model_name)
    l1_errors = []
    l2_errors = []
    names = []
    for file_name in os.listdir(model_predictions_dir):
        name = file_name.split('.')[0]
        print(f'Processing {name}')
        # read hdf5 file
        h5_file = h5py.File(os.path.join(gt_large, f'{name}.hdf5'), 'r')
        # read sdf
        sdf_128 = h5_file['128_sdf'][:]

        # read nvdb
        grid, features, _ = fvdb.load(os.path.join(model_predictions_dir, f'{name}.nvdb'))

        ijk = grid.ijk.jdata.numpy()
        ijk =  ijk + (128//2)
        ijk = np.clip(ijk, 0, 128)  # Ensure indices are within bounds
        ijk = ijk.astype(int)
        features = features.jdata.numpy()[:, 0]  
        actual_sdf = sdf_128[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        actual_sdf = actual_sdf* 32
        l1_error = np.mean(np.abs(features - actual_sdf))
        l2_error = np.mean((features - actual_sdf)**2)
        # print(f'L1 error: {l1_error}, L2 error: {l2_error}')

        l1_errors.append(l1_error)
        l2_errors.append(l2_error)
        names.append(name)
        # print(f'L1 error: {l1_error}, L2 error: {l2_error}')

    import pandas as pd
    df_sdf_errors = pd.DataFrame({
        'name': names,
        'l1_error': l1_errors,
        'l2_error': l2_errors
    })
    df_describe = df_sdf_errors.describe()
    print(df_describe)
    return df_describe

In [4]:
# df_describe = eval_sdf('40_rerun_31_CNN_Upsampler_Manual_NMC_data_l1_&_sign_loss', '2')

In [5]:
# df_describe

In [6]:
# df_describe

In [14]:
def display_model_predictions(file_name, model_name, sub_dir=None, display_sdf=False):
    gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    test_predictions_dir = '/data/workspaces/spanwar/results/ssu/test_predictions'
    if sub_dir is not None:
        model_name = os.path.join(model_name, sub_dir)
    model_predictions_dir = os.path.join(test_predictions_dir, model_name)

    name = file_name.split('.')[0]
    print(f'Processing {name}')
    # read hdf5 file
    h5_file = h5py.File(os.path.join(gt_large, f'{name}.hdf5'), 'r')
    # read sdf
    sdf_32 = h5_file['32_sdf'][:]
    sdf_128 = h5_file['128_sdf'][:]

    # read nvdb
    grid, features, _ = fvdb.load(os.path.join(model_predictions_dir, f'{name}.nvdb'))

    v, f, _ = grid.marching_cubes(features)
    vn = v.jdata.cpu().numpy()
    fn = f.jdata.cpu().numpy()

    if display_sdf:
        print(f'input sdf_32 {name}')
        v32, f32, _, _ = measure.marching_cubes(sdf_32, level=0.0)
        plot(v32, f32)
        v, f,_, _ = measure.marching_cubes(sdf_128, level=0.0)
        print(f'Plotting ground truth for {name}')
        plot(v, f)
    print(f'Plotting predictions for {name}')
    plot(vn, fn)

In [15]:
display_model_predictions('00000020',
                          '40_rerun_31_CNN_Upsampler_Manual_NMC_data_l1_&_sign_loss', 
                          '2', 
                          display_sdf=True)

Processing 00000020
input sdf_32 00000020


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(16.0, 16.…

Plotting ground truth for 00000020


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(64.0, 64.…

Plotting predictions for 00000020


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.000239…

In [23]:
display_model_predictions('00000020',
                          '50_rerun_49_with_300_epochs', 
                          None, 
                          display_sdf=False)

Processing 00000020
Plotting predictions for 00000020


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4995434…

In [24]:
display_model_predictions('00000020',
                          '49_random_direction_per_voxel', 
                          None, 
                          display_sdf=False)

Processing 00000020
Plotting predictions for 00000020


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4994879…

# Compelitly Random Vector Training

In [ ]:
def gt_normalize_vertices(vertices):
    """
    Normalize vertices and optionally return normalization parameters
    """
    #normalize diagonal=1
    x_max = np.max(vertices[:,0])
    y_max = np.max(vertices[:,1])
    z_max = np.max(vertices[:,2])
    x_min = np.min(vertices[:,0])
    y_min = np.min(vertices[:,1])
    z_min = np.min(vertices[:,2])
    x_mid = (x_max+x_min)/2
    y_mid = (y_max+y_min)/2
    z_mid = (z_max+z_min)/2
    x_scale = x_max - x_min
    y_scale = y_max - y_min
    z_scale = z_max - z_min
    model =  torch.load(path)scale = np.sqrt(x_scale*x_scale + y_scale*y_scale + z_scale*z_scale)
    
    vertices[:,0] = (vertices[:,0]-x_mid)/scale
    vertices[:,1] = (vertices[:,1]-y_mid)/scale
    vertices[:,2] = (vertices[:,2]-z_mid)/scale

    return vertices

In [4]:
# Add random rotation
def random_rotation_matrix(gt_mesh):
    # Generate random rotation matrix
    angles = np.random.uniform(0, 2*np.pi, 3)  # Random angles for x, y, z
    
    # Rotation matrices for each axis
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(angles[0]), -np.sin(angles[0])],
                   [0, np.sin(angles[0]), np.cos(angles[0])]])
    
    Ry = np.array([[np.cos(angles[1]), 0, np.sin(angles[1])],
                   [0, 1, 0],
                   [-np.sin(angles[1]), 0, np.cos(angles[1])]])
    
    Rz = np.array([[np.cos(angles[2]), -np.sin(angles[2]), 0],
                   [np.sin(angles[2]), np.cos(angles[2]), 0],
                   [0, 0, 1]])
    
    R = Rz @ Ry @ Rx
    gt_mesh.vertices = gt_mesh.vertices @ R.T
    return gt_mesh


## Data Processing

In [5]:
%load_ext autoreload
%autoreload 2
import os
import sys
import trimesh
import fvdb
import argparse
from pprint import pprint
import tqdm
import h5py
from skimage import measure
from meshplot import plot
from scipy.spatial.distance import cdist
import joblib
import numpy as np
from scipy.spatial import KDTree
import igl
import fvdb.nn as fvnn

# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
import eval
import utils
import models
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 
from utils import mesh_tools as mt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
%%time
import os
import time
import torch
import random
def scaled_sdf(threshold, sdf_arr: np.array):
    '''scales the SDF array by the threshold value'''
    return (threshold-1)*sdf_arr[:, None]

def get_sdf_sampled_data(obj_name, threshold=65, grid_size = 33, random_type='uniform'):
    output_path = '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random'
    file_name = obj_name.split('.')[0]

    _dir_gt = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs'
    # find the .obj file in the directory by walking through the directory
    file_dir_gt = os.path.join(_dir_gt, obj_name.split('.')[0])
    obj_files = [f for f in os.listdir(file_dir_gt) if f.endswith('.obj')]
    file_gt = os.path.join(file_dir_gt, obj_files[0])
    
    # get sdf from the point
    gt_mesh = trimesh.load(file_gt, process=False)
    gt_mesh.vertices = gt_normalize_vertices(gt_mesh.vertices)
    
    gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    # read hdf5 file
    filename = obj_name.split('.')[0]
    h5_file = h5py.File(os.path.join(gt_large, f'{filename}.hdf5'), 'r')

    
    sdf_32 = h5_file['32_sdf'][:]
    mask_sdf_32 = mt.make_mask_close(sdf_32, threshold)
    grid_original = mt.mesh_grid(grid_size, normalize=False)
    mask_sdf_32 = mask_sdf_32.flatten()
    grid_original = grid_original[mask_sdf_32]
    m3g = mt.mesh_grid(3)-1

    # random select a random direction for each voxel
    num_elements = grid_original.shape[0]
    random_indices = torch.randint(0, m3g.shape[0], (num_elements,))
    
    # random number between 0 to 1
    if random_type=='nonUniform':
        rand_values = np.random.rand(num_elements, 3)
    elif random_type=='uniform':
        rand_values = np.random.rand(num_elements, 1)
    elif random_type=='randomChoice':
        my_list = [-1, -.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1]
        rand_values = random.choice(my_list)

    # create new grid    
    grid_new = grid_original + m3g[random_indices[0]] * (rand_values/2)
    grid_new = np.clip(grid_new, 0, grid_size-1)
    # print(m3g[random_indices])
    # print(rand_values)
    # voxel vector
    voxel_vector = (grid_new - grid_original)*2
    # print('voxel vector')
    # print(voxel_vector)
    # print(grid_new)
    # print(grid_original)
    # normalize grid
    grid_norm = grid_new/(grid_size-1)-0.5

    # sdf calculation
    sdf_val = igl.signed_distance(grid_norm, gt_mesh.vertices, gt_mesh.faces)[0]
    sdf_val_array = np.full((grid_size, grid_size, grid_size), 100.0)
    sdf_val_array[grid_original[:, 0], grid_original[:, 1], grid_original[:, 2]] = sdf_val

    grid_original = torch.tensor(grid_original, dtype=torch.int, device='cpu')

    # fvdb tensors
    fvdb_grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(grid_original),
                                        voxel_sizes=(1/(grid_size-1)),
                                   origins=torch.tensor([0, 0, 0], device='cpu')
                                   )
    ijk = fvdb_grid.ijk.jdata
    sdf_val = sdf_val_array[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    sdf_val = torch.tensor(sdf_val, dtype=torch.float32, device='cpu')
    sdf_val = scaled_sdf(threshold, sdf_val)

    sdf_val_output = sdf_32[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    sdf_val_output = torch.tensor(sdf_val_output, dtype=torch.float32, device='cpu')
    sdf_val_output = scaled_sdf(threshold, sdf_val_output)

    # input feature
    voxel_vector = torch.tensor(voxel_vector, dtype=torch.float32, device='cpu')
    sdf_val_cat = torch.cat([sdf_val_output, voxel_vector], dim=-1)
    # print(sdf_val_cat)

    input_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(sdf_val_cat))
    output_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(sdf_val))

    # save the input and output VDB tensors
    input_vdb_path = os.path.join(output_path, 
                                  f'{file_name}_input.nvdb')
    output_vdb_path = os.path.join(output_path, 
                                   f'{file_name}_output.nvdb')

    fvdb.save(input_vdb_path, input_tensor.grid, input_tensor.data, compressed=True)
    fvdb.save(output_vdb_path, output_tensor.grid, output_tensor.data, compressed=True)

    # nv, nf, _ = input_tensor.grid.marching_cubes(input_tensor.data)
    # plot(nv.jdata.numpy(), nf.jdata.numpy())

    # print(input_tensor.data.jdata.shape)
    # print(output_tensor.data.jdata.shape)
    # nv, nf, _ = output_tensor.grid.marching_cubes(output_tensor.data)
    # plot(nv.jdata.numpy(), nf.jdata.numpy())
    # return {'obj_name': obj_name, 'input_tensor': input_tensor, 'output_tensor': output_tensor}

get_sdf_sampled_data('00000020.obj')

CPU times: user 681 ms, sys: 60 ms, total: 741 ms
Wall time: 798 ms


In [5]:
m3g = mt.mesh_grid(3)-1
np.random.randint(0, m3g.shape[0]), m3g.shape

(12, (27, 3))

In [6]:
# %%time
# mt.mesh_grid(20)

In [22]:
pred_dir = '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random'
grid, feature, _ = fvdb.load(os.path.join(pred_dir, '00006574_input.nvdb'), device='cpu')
print(feature.jdata)
feature.jdata = feature.jdata[:, 0].unsqueeze(1)
nv, nf, _ = grid.marching_cubes(feature)
plot(nv.jdata.numpy(), nf.jdata.numpy())

tensor([[ 2.0951,  0.5000, -1.0000,  0.5000],
        [ 1.8638,  1.0000,  0.5000, -1.0000],
        [ 1.8516,  1.0000,  0.5000, -1.0000],
        ...,
        [ 2.6135,  0.0000,  0.5000, -0.5000],
        [ 2.6135,  0.0000, -0.5000,  0.5000],
        [ 2.6515,  0.0000, -0.5000,  0.0000]])


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5000000…

In [23]:
pred_dir = '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random'
grid, feature, _ = fvdb.load(os.path.join(pred_dir, '00000020_output.nvdb'), device='cpu')
print(feature.jdata)
feature.jdata = feature.jdata[:, 0].unsqueeze(1)
nv, nf, _ = grid.marching_cubes(feature)
plot(nv.jdata.numpy(), nf.jdata.numpy())

tensor([[2.2905],
        [2.3236],
        [2.3020],
        ...,
        [2.8969],
        [3.2385],
        [3.1639]])


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4914280…

In [7]:
# Imports
import os
import sys
sys.path.append('../src/utils')
sys.path.append('../src/data_utils')
import numpy as np
from torch.utils.data import Dataset
from tqdm import tqdm
import h5py
import joblib
import torch
import fvdb.nn as fvnn
import mesh_tools as mt
import fvdb_utils as fu

class ABCDataset(Dataset):
    def __init__(self, src_dir,
                 names_set,
                 dataset_grids,
                 input_size,
                 mask_threshold, # in term grid size i,e grid size = 32 -> 3/32
                #  is_crop,
                #  crops_size,
                #  crops_size_probability,
                #  crops_threshold,
                 upsample_factor,
                 unique_random_direction, 
                 max_tries=100, 
                 is_test=False,
                 n_jobs=-1):
        
        self.input_dir = src_dir
        self.names_set = names_set
        self.dataset_grids = dataset_grids
        self.input_size = input_size
        self.mask_threshold = mask_threshold

        self.upsample_factor = upsample_factor #4
        self.unique_random_direction = unique_random_direction

        # self.is_crop = is_crop
        # self.crops_size = crops_size
        # self.crops_size_probability = crops_size_probability
        # self.crops_threshold = crops_threshold

        self.max_tries = max_tries
        self.is_test = is_test
        self.n_jobs = n_jobs

        # helping class
        self.sdfToVDB = fu.sdfToVDB(threshold=self.mask_threshold)

        # stepup to read the dataset
        self._read_dataset()  # This will run setup() and read the files in parallel


    def _get_all_shifted_positions(self, vdb_tensor, size, upsample_factor):
        m3g = torch.tensor(mt.mesh_grid(upsample_factor+1), device=vdb_tensor.device) - (upsample_factor//2)

        new_ijks = []
        new_features = []
        for mg in m3g:
            ijk = vdb_tensor.grid.ijk.jdata
            ijk = (upsample_factor * ijk + mg).view(-1, 3)
            ijk = np.clip(ijk.cpu().detach().numpy(), 0, (size-1)*upsample_factor)
            ijk_vector = ijk - (vdb_tensor.grid.ijk.jdata.cpu().detach().numpy() * upsample_factor)
            ijk_vector = ijk_vector / (upsample_factor // 2)  # Normalize to values between -1 and 1
            ijk_vector = torch.tensor(ijk_vector, dtype=torch.float32, device=vdb_tensor.device)

            new_features.append(torch.cat([vdb_tensor.data.jdata, ijk_vector], axis=-1))
            new_ijks.append(torch.tensor(ijk, dtype=torch.int, device=vdb_tensor.device))
        return new_features, new_ijks
    
    def _get_item(self, obj_name):
        '''Read the SDF in h5 file.'''

        sdf_dict = {}
        path = os.path.join(self.input_dir, obj_name)
        
        with h5py.File(path, 'r') as f:
            # check if the file has the required datasets
            if '32_sdf' not in f or '64_sdf' not in f or '128_sdf' not in f:
                raise ValueError(f"File {path} does not contain required datasets.")
            
            # fetch the SDF and output SDF
            sdf_dict['obj_name'] = obj_name
            for grid_size in self.dataset_grids:
                sdf_dict[grid_size+1] = f[f'{grid_size}_sdf'][:]
        
        return sdf_dict

    def _cropped_mask(self, mask):
        """
        Randomly crop a 3D array so that the crop contains at least n nonzero elements.
        crop_size: int or tuple (crop_x, crop_y, crop_z)
        threshold: minimum number of nonzero elements required in the crop
        max_tries: maximum number of attempts
        """

        # _crop_size = np.random.choice(self.crops_size, p=self.crops_size_probability)
        # _crop_size = int(_crop_size)
        _crop_size = self.crops_size
        if isinstance(_crop_size, int):
            crop_size = (_crop_size, _crop_size, _crop_size)
        sx, sy, sz = mask.shape
        cx, cy, cz = crop_size

        for _ in range(self.max_tries):
            x = np.random.randint(0, sx - cx + 1)
            y = np.random.randint(0, sy - cy + 1)
            z = np.random.randint(0, sz - cz + 1)
            
            crop = mask[x:x+cx, y:y+cy, z:z+cz]
            crop_threshold = 400

            if np.count_nonzero(crop) >= crop_threshold:
                mask_crop = np.zeros_like(mask, dtype=bool)
                mask_crop[x:x+cx, y:y+cy, z:z+cz] = crop
                return mask_crop
            
        # Ignore threshold - just return the crop
        print(f"Warning: Could not find a valid crop after {self.max_tries} attempts. Returning the last attempt.")
        mask_crop = np.zeros_like(mask, dtype=bool)
        mask_crop[x:x+cx, y:y+cy, z:z+cz] = crop
        return mask_crop

    def _read_dataset(self):
        # out = joblib.Parallel(n_jobs=self.n_jobs)(joblib.delayed(get_sdf_sampled_data)
        #                                           (obj_name) for obj_name in tqdm(self.names_set))
        if self.is_test:
            out = joblib.Parallel(n_jobs=self.n_jobs)(joblib.delayed(self._get_item)
                                                    (obj_name) for obj_name in tqdm(self.names_set))

            # check for empty set
            if len(out) == 0:
                raise ValueError("No valid SDF data found in the provided dataset.")
            
            # mask SDFs of 32
            self.mask_32s = [mt.make_mask_close(_dict[33], self.mask_threshold)  for _dict in out]
        
        else:
            self.out = [{'obj_name':  name} for name in self.names_set]

    def _get_vdb_from_sdf(self, index):
        _dict = self.out[index]

        # create a set to hold the vdb tensors
        vdb_set = []
        vdb_set.append(_dict['obj_name'])
        if not self.is_test:
            # input_vdb, output_vdb = self
            # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            output_path = '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random'
            file_name = _dict["obj_name"].split('.')[0]
            grid, features, _ = fvdb.load(os.path.join(output_path, f'{file_name}_input.nvdb'))
            input_tensor = fvnn.VDBTensor(grid, features)
            grid, features, _ = fvdb.load(os.path.join(output_path, f'{file_name}_output.nvdb'))
            output_tensor = fvnn.VDBTensor(grid, features)

            vdb_set.append(input_tensor)
            vdb_set.append(output_tensor)
            return tuple(vdb_set)

        # crop mask SDFs of 32
        mask_32 = self.mask_32s[index]
        # if self.is_crop:
        #     mask_32_index = self._cropped_mask(mask_32_index)

        # check it it a test set
        if self.is_test:
            # create a mask for the test set
            vdb_tensor = self.sdfToVDB.sdf_to_vdb(
                                        sdf_arr=_dict[33],
                                        large_sdf_arr=None,
                                        mask=mask_32,
                                        upsample_factor=None,
                                        unique_random_direction=None,
                                        size=self.input_size, #33
                                        is_test=True
                                    )

            new_features, new_ijks = self._get_all_shifted_positions(vdb_tensor, 
                                            size=self.input_size, 
                                            upsample_factor=self.upsample_factor)

            vdb_set.append(vdb_tensor)
            vdb_set.append(new_ijks)
            vdb_set.append(new_features)
            vdb_set.append(_dict[129]*(self.mask_threshold-1)) # scale the SDF by the threshold value
            return tuple(vdb_set)

        input_vdb, output_vdb = self.sdfToVDB.sdf_to_vdb(_dict[33],
                                                        _dict[129],
                                                        mask_32,
                                                        size=self.input_size, # 33
                                                        upsample_factor=self.upsample_factor,
                                                        unique_random_direction=self.unique_random_direction)

        # add the input vdb to the set
        vdb_set.append(input_vdb)

        vdb_set.append(output_vdb)                                 

        return tuple(vdb_set)

    def __len__(self):
        return len(self.out)
    
    def __getitem__(self, index):
        return self._get_vdb_from_sdf(index)


class ABCDataLoader():
    def __init__(self, 
                 input_dir, 
                 config,
                 n_samples=None):
        self.input_dir = input_dir
        self.config = config
        self.n_samples = n_samples

    @staticmethod
    def custom_collate_fn(batch):
        # batch is a list of tuples: [(vdb_32, vdb_64, vdb_128), ...]
        # level 2: two vdbs
        # level 3: three vdbs
        # level 4: four vdbs
        level = len(batch[0])-1 # -1 because first element is obj_name
        if level == 2:
            obj_names, vdb_1s, vdb_2s = zip(*batch)
            return list(obj_names), list(vdb_1s), list(vdb_2s)
        elif level == 3:
            obj_names, vdb_1s, vdb_2s, vdb_3s = zip(*batch)
            return list(obj_names), list(vdb_1s), list(vdb_2s), list(vdb_3s)
        elif level == 4:
            obj_names, vdb_1s, vdb_2s, vdb_3s, vdb_4s = zip(*batch)
            return list(obj_names), list(vdb_1s), list(vdb_2s), list(vdb_3s), list(vdb_4s)
        else:
            raise ValueError(f"Unsupported upscaling (too many objects): workable upscaling are 64, 128, 256, not above 256")
        
    @staticmethod
    def custom_collate_fn_test(batch):
        # batch is a list of tuples: [(vdb_tensor, new_ijks, new_features), ...]
        obj_names, vdb_tensors, new_ijks, new_features, actual_sdf = zip(*batch)
        return list(obj_names), list(vdb_tensors), list(new_ijks), list(new_features), list(actual_sdf)

    def get_vdb_data_loaders(self,
                             train_dataset,
                             val_dataset,
                             test_dataset, 
                             batch_size=1, 
                             shuffle=None, 
                             num_workers=0):
        
        is_eval = False  # This can be set based on your evaluation mode
        if not is_eval:
            train_dataloader =  torch.utils.data.DataLoader(train_dataset, 
                                                collate_fn=self.custom_collate_fn,
                                                batch_size=batch_size, 
                                                shuffle=True, 
                                                num_workers=num_workers)
            val_dataloader = torch.utils.data.DataLoader(val_dataset,
                                                collate_fn=self.custom_collate_fn,
                                                batch_size=batch_size,
                                                shuffle=True, 
                                                num_workers=num_workers)
        else:
            train_dataloader = None
            val_dataloader = None

        test_dataloader = torch.utils.data.DataLoader(test_dataset,
                                            collate_fn=self.custom_collate_fn_test,
                                            # batch_size=batch_size,
                                            batch_size=1,  # Test loader usually has batch size of 1
                                            shuffle=False, 
                                            num_workers=num_workers)
        return train_dataloader, val_dataloader, test_dataloader


    def split_dataset(self,
                      names_set, 
                      train_ratio=0.6, 
                      val_ratio=0.2):
        """
        Splits the dataset into train, validation, and test sets.
        """
        total_size = len(names_set)
        train_size = int(total_size * train_ratio)
        val_size = int(total_size * val_ratio)

        np.random.shuffle(names_set)
        train_set = names_set[:train_size]
        val_set = names_set[train_size:train_size + val_size]
        test_set = names_set[train_size + val_size:]

        print(f"Dataset split: {len(train_set)} train, {len(val_set)} val, {len(test_set)} test")

        return train_set, val_set, test_set

    
    def get(self, names_set):
        if self.n_samples is not None:
            if not isinstance(self.n_samples, int):
                raise ValueError("n_samples must be an integer or None")
            names_set = names_set[:self.n_samples]

        train_set, val_set, test_set = self.split_dataset(names_set, 
                                        train_ratio=0.6, 
                                        val_ratio=0.2)
        
        # check correct spliting
        # with open('test_names_file.txt', 'r') as f:
        #     test_set_from_file = set(f.read().splitlines())
        # assert set(test_set) == test_set_from_file, "Test set does not match the file content"

        is_eval = False
        if not is_eval:
            train_dataset = ABCDataset(
                src_dir=self.input_dir,
                names_set=train_set,
                dataset_grids=self.config['data']['dataset_grids'],
                input_size=self.config['data']['input_size'],
                mask_threshold=self.config['data']['mask_threshold'],
                # is_crop=self.config['data']['is_crop']['train'],
                # crops_size=self.config['data']['crops_size'],
                # crops_size_probability=self.config['data']['crops_size_probability'],
                # crops_threshold=self.config['data']['crops_threshold'],
                upsample_factor=self.config['data']['upsample_factor'],
                unique_random_direction='uniform',
                n_jobs=-1
            )
            val_dataset = ABCDataset(
                src_dir=self.input_dir,
                names_set=val_set,
                dataset_grids=self.config['data']['dataset_grids'],
                input_size=self.config['data']['input_size'],
                mask_threshold=self.config['data']['mask_threshold'],
                # is_crop=self.config['data']['is_crop']['val'],
                # crops_size=self.config['data']['crops_size'],
                # crops_size_probability=self.config['data']['crops_size_probability'],
                # crops_threshold=self.config['data']['crops_threshold'],
                upsample_factor=self.config['data']['upsample_factor'],
                unique_random_direction='uniform',
                n_jobs=-1
            )
        else:
            train_dataset = None
            val_dataset = None
            
        test_dataset = ABCDataset(
            src_dir=self.input_dir,
            names_set=test_set,
            dataset_grids=self.config['data']['dataset_grids'],
            input_size=self.config['data']['input_size'],
            mask_threshold=self.config['data']['mask_threshold'],
            # is_crop=self.config['data']['is_crop']['test'],
            # crops_size=self.config['data']['crops_size'],
            # crops_size_probability=self.config['data']['crops_size_probability'],
            # crops_threshold=self.config['data']['crops_threshold'],
            upsample_factor=self.config['data']['upsample_factor'],
            unique_random_direction='uniform',
            is_test=True,
            n_jobs=-1
        )
        
        train_dataloader, val_dataloader, test_dataloader = self.get_vdb_data_loaders(
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            batch_size=16,
            shuffle=True,
            num_workers=8
        )
        return train_dataloader, val_dataloader, test_dataloader

In [8]:
st.set_reproducibility(is_reproducible=True,
                           seed=42)
names_set = os.listdir('/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt')
# names_set = names_set[:10]
train_ratio = 0.6
val_ratio = 0.2
total_size = len(names_set)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)

np.random.shuffle(names_set)
train_set = names_set[:train_size]
val_set = names_set[train_size:train_size + val_size]
test_set = names_set[train_size + val_size:]
# check if test set is correctly split
# with open('test_names_file.txt', 'r') as f:
#     test_set_from_file = f.read().splitlines()
# assert set(test_set) == set(test_set_from_file), "Test set does not match the expected test set."


Setting reproducibility with seed: 42


In [9]:
%%time
def run_data_processing(names):
    joblib.Parallel(n_jobs=-1)(joblib.delayed(get_sdf_sampled_data)(name) for name in tqdm(names))
run_data_processing(train_set + val_set)

  0%|          | 0/4280 [00:00<?, ?it/s]

 29%|██▉       | 1248/4280 [00:41<01:41, 29.84it/s]/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 4280/4280 [03:08<00:00, 22.76it/s]


CPU times: user 58.2 s, sys: 1.41 s, total: 59.6 s
Wall time: 4min


In [10]:
# load data
import read_config
config = read_config.read_yaml_config('config_51_14082025_1100.yaml')
import random
st.set_reproducibility(is_reproducible=True,
                           seed=42)
input_dir = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
names_set = os.listdir('/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt')
(train_dataloader, 
val_dataloader, 
test_dataloader) = ABCDataLoader(
                                    input_dir=input_dir,
                                    config=config,
                                    # n_samples=10
                                ).get(names_set=names_set)

Setting reproducibility with seed: 42
Dataset split: 3210 train, 1070 val, 1071 test


100%|██████████| 1071/1071 [00:28<00:00, 37.33it/s]



In [11]:
# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
from eval import ABC_eval
from models import unet
from models import model as simpleModels
from models import unet as unetModels
from training import model_training
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 

In [12]:
config['logging']

True

In [13]:
logger = wandb_logging.WandbLogger(
                        logging=False,
                        project_name=config['wandb']['project_name'],
                        entity=config['wandb']['entity'],
                        name=config['wandb']['name'],
                        group=config['wandb']['group'],
                        tags=config['wandb']['tags'],
                        notes=config['wandb']['notes'],
                        config=config['wandb']['config'],
                    )

In [14]:
import os
import sys
import fvdb
import torch
import wandb
import fvdb.nn as fvnn
import numpy as np
from tqdm import tqdm
import pandas as pd
sys.path.append('../src/training')
from loss import LossFunctions

sys.path.append('../src/utils')
from ssu_tools import positional_encoding
import mesh_tools as mt

# parent_path = os.path.abspath(os.path.join(__file__, "../../../flow_matching"))
# print(f"Parent path: {parent_path}")
# sys.path.append(parent_path)
sys.path.append('../flow_matching')
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
# from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper


In [15]:

class ModelTrainer:
    def __init__(self,
                 model_name, 
                 model, 
                 num_epochs,
                 train_loader, 
                 val_loader,
                 test_loader,
                 upsample_factor,
                 input_size,
                #  pos_enc_dim, 
                 optimizer, 
                 loss_fn_name,
                 is_save_model,
                 save_model_dir,
                 save_predictions_dir, 
                 logger
                 ):
        
        self.model_name = model_name
        self.model = model
        self.num_epochs = num_epochs

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.upsample_factor = upsample_factor
        self.input_size = input_size
        # self.pos_enc_dim = pos_enc_dim

        self.optimizer = optimizer
        self.loss_fn = loss_fn_name
        self.loss_fn = LossFunctions(loss_fn_name).loss_fn
        
        self.is_save_model = is_save_model
        self.save_model_dir = save_model_dir
        self.save_predictions_dir = save_predictions_dir
        
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)

        self.logger = logger
        # self.logger.log({'loss_weights': wandb.Table(data=[self.loss_weights], 
                                                    #  columns=['w1', 'w2', 'w3'])})

    def train_sub_step(self, inputs):
        # inputs = positional_encoding(inputs, self.pos_enc_dim)
        outputs = self.model(inputs)
        return outputs

    def append_feature(self, input_vdbs, feature):
        concat_features = torch.cat([input_vdbs.jdata, feature], dim=-1)
        return fvnn.VDBTensor(input_vdbs.grid, input_vdbs.grid.jagged_like(concat_features))

    def sample_fm(self, input_vdbs, output_vdbs):
        t = torch.rand_like(output_vdbs.jdata, device=self.device)
        # print(t.shape, input_vdbs.jdata[:, 0].shape, )
        features = t * input_vdbs.jdata[:, 0].unsqueeze(1) + (1 - t) * output_vdbs.jdata
        features = torch.cat([features, input_vdbs.jdata[:, 1:], t], dim=-1)
        # print(features.shape)
        sampled_vdbs = fvnn.VDBTensor(input_vdbs.grid,
                                      input_vdbs.grid.jagged_like(features))
        return sampled_vdbs

    def velocity_fm(self, input_vdbs, output_vdbs):
        velocity = output_vdbs.jdata[:, 0] - input_vdbs.jdata[:, 0]
        velocity_vdb = fvnn.VDBTensor(input_vdbs.grid,
                                  input_vdbs.grid.jagged_like(velocity[:, None]))
        return velocity_vdb
    
    def eval_fm_steps(self, input_vdbs, model, n_steps=4):
        dt = 1 / n_steps
        t = torch.full_like(input_vdbs.jdata[:, 0], 0).to(self.device)
        t = t.unsqueeze(1)  # Ensure t is a column vector
        input_vdbs = self.append_feature(input_vdbs, t)
        for t in range(n_steps):
            t = torch.full_like(input_vdbs.jdata[:, 0], t/n_steps).to(self.device)
            # t = t.unsqueeze(1)  # Ensure t is a column vector
            input_vdbs.jdata[:, -1] = t
            updated_sdf = input_vdbs.jdata[:, 0].unsqueeze(1) + dt * model(input_vdbs).jdata
            input_vdbs.jdata[:, 0] = updated_sdf.squeeze(1)
            # print(input_vdbs.jdata.shape)
            input_vdbs = fvnn.VDBTensor(input_vdbs.grid,
                                        input_vdbs.grid.jagged_like(
                                            input_vdbs.jdata
                                        ))
            
        predicted_vdbs = fvnn.VDBTensor(input_vdbs.grid,
                                        input_vdbs.grid.jagged_like(
                                            input_vdbs.jdata[:, 0].unsqueeze(1)
                                        ))
        return predicted_vdbs
    
    def meta_lib_fm(self, input_vdbs, output_vdbs):
        path = AffineProbPath(scheduler=CondOTScheduler())
        t = torch.rand(output_vdbs.jdata.shape[0]).to(self.device)

        path_sample = path.sample(t=t, 
                                  x_0=input_vdbs.jdata[:,0], 
                                  x_1=output_vdbs.jdata[:,0])
        xt = path_sample.x_t
        t = path_sample.t
        velocity = path_sample.dx_t

        xt_feature = torch.cat([xt.unsqueeze(1), 
                                input_vdbs.jdata[:, 1:], 
                                t.unsqueeze(1)], dim=-1)
        
        xt = fvnn.VDBTensor(input_vdbs.grid,
                            input_vdbs.grid.jagged_like(xt_feature))
        velocity = fvnn.VDBTensor(input_vdbs.grid,
                                    input_vdbs.grid.jagged_like(velocity[:, None]))
        return xt, velocity

    def train(self):
        min_val_loss = float('inf')
        for epoch in range(self.num_epochs):
            self.model.train()
            total_loss = 0
            if (epoch+1) % 3 == 0:
                run_data_processing(train_set + val_set)

            for batch in tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.num_epochs}'):
                obj_names, vdb_input, vdb_output = batch

                vdb_inputs = fvdb.jcat(vdb_input)
                vdb_outputs = fvdb.jcat(vdb_output)
                vdb_inputs = vdb_inputs.cuda()
                vdb_outputs = vdb_outputs.cuda()
                self.optimizer.zero_grad()

                # manual flow matching steps
                # xt = self.sample_fm(vdb_inputs, vdb_outputs)
                # velocity = self.velocity_fm(vdb_inputs, vdb_outputs)

                # flow matching steps using meta-lib
                xt, velocity = self.meta_lib_fm(vdb_inputs, vdb_outputs)

                preds = self.train_sub_step(xt)

                # Compute losses for each output and target
                loss = self.loss_fn(preds.jdata, velocity.jdata)

                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
            avg_loss = total_loss / len(self.train_loader)
            print(f"Epoch {epoch+1}/{self.num_epochs}, Loss: {avg_loss:.4f}")
            if self.val_loader:
                (avg_val_loss) = self.validation()
            
            # Log the training loss
            self.logger.log({
                'train_loss': avg_loss,
                'val_loss': avg_val_loss,
                'epoch': epoch + 1
            })
            
            # Check if validation loss is lower than the minimum recorded loss
            if avg_val_loss < min_val_loss:
                min_val_loss = avg_val_loss
                if self.is_save_model:
                    self.save_model()
        
        print(f"Training complete. Minimum validation loss: {min_val_loss:.4f}")

        # if self.is_save_model:
        #     print(f'Saving the predictions to {self.save_predictions_dir}')
        #     self.save_predictions()
        
    def validation(self):
        self.model.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validation'):
                obj_names, vdb_input, vdb_output = batch
                vdb_inputs = fvdb.jcat(vdb_input)
                vdb_outputs = fvdb.jcat(vdb_output)
                vdb_inputs = vdb_inputs.cuda()
                vdb_outputs = vdb_outputs.cuda()

                # preds = self.train_sub_step(vdb_inputs)
                preds = self.eval_fm_steps(vdb_inputs, self.model, n_steps=4)

                loss = self.loss_fn(preds.jdata, vdb_outputs.jdata)

                total_loss += loss.item()
                
        avg_loss = total_loss / len(self.val_loader)
        print(f"Validation Loss: {avg_loss:.4f}")
        return avg_loss

    def save_model(self):
        path = os.path.join(self.save_model_dir, f"{self.model_name}.pth")
        torch.save(self.model, path)
        print(f"Model saved to {path}")

    # def get_all_shifted_positions(self, vdb_tensor, size, upsample_factor):
    #     m3g = torch.tensor(mt.mesh_grid(upsample_factor+1), device=vdb_tensor.device) - (upsample_factor//2)

    #     new_ijks = []
    #     new_features = []
    #     for mg in m3g:
    #         ijk = vdb_tensor.grid.ijk.jdata
    #         ijk = (upsample_factor * ijk + mg).view(-1, 3)
    #         ijk = np.clip(ijk.cpu().detach().numpy(), 0, (size-1)*upsample_factor)
    #         ijk_vector = ijk - (vdb_tensor.grid.ijk.jdata.cpu().detach().numpy() * upsample_factor)
    #         ijk_vector = ijk_vector / (upsample_factor // 2)  # Normalize to values between -1 and 1
    #         ijk_vector = torch.tensor(ijk_vector, dtype=torch.float32, device=vdb_tensor.device)

    #         new_features.append(torch.cat([vdb_tensor.data.jdata[:, 0][:, None], ijk_vector], axis=-1))
    #         new_ijks.append(torch.tensor(ijk, dtype=torch.int, device=vdb_tensor.device))
    #     return new_features, new_ijks
    
    def test_fm_steps(self, input_vdb, model, n_steps):
        dt = 1 / n_steps
        t = torch.full_like(input_vdb.jdata[:, 0], 0).to(input_vdb.device)
        t = t.unsqueeze(1)
        input_vdb = self.append_feature(input_vdb, t)
        for t in range(n_steps):
            t = torch.full_like(input_vdb.jdata[:, 0], float(t)/n_steps).to(input_vdb.device)
            input_vdb.jdata[:, -1] = t
            updated_sdf = input_vdb.jdata[:, 0].unsqueeze(1) + dt * model(input_vdb).jdata
            input_vdb.jdata[:, 0] = updated_sdf.squeeze(1)
            input_vdb = fvnn.VDBTensor(input_vdb.grid,
                                        input_vdb.grid.jagged_like(
                                            input_vdb.jdata
                                        ))

        predicted_vdbs = fvnn.VDBTensor(input_vdb.grid,
                                        input_vdb.grid.jagged_like(
                                            input_vdb.jdata[:, 0].unsqueeze(1)
                                        ))
        return predicted_vdbs

    def predictions_fm_steps(self, 
                             input_vdb, 
                             new_features, 
                             new_ijks, 
                             model, 
                             n_steps,
                             actual_sdf=None):
        # new_features, new_ijks = self.get_all_shifted_positions(input_vdb, 
        #                                                         size=self.input_size, 
        #                                                         upsample_factor=self.upsample_factor)
        all_inputs = []
        for feature in new_features:
            all_inputs.append(fvnn.VDBTensor(input_vdb.grid,
                                            input_vdb.grid.jagged_like(feature)))
        all_inputs_vdb = fvdb.jcat(all_inputs)

        upsampled_sdf_size = ((self.input_size - 1) * self.upsample_factor) + 1
        sdf = np.full((upsampled_sdf_size, 
                       upsampled_sdf_size, 
                       upsampled_sdf_size), 100.0)
        
        # for shifted_feature, shifted_ijk in zip(new_features, new_ijks):
        #     new_vdb_tensor = fvnn.VDBTensor(input_vdb.grid,
        #                                     input_vdb.grid.jagged_like(shifted_feature))
        #     pred = self.test_fm_steps(new_vdb_tensor, model, n_steps)

        #     ijk = shifted_ijk.cpu().detach().numpy()
        #     pred_values = pred.jdata.cpu().detach().numpy().squeeze()  # Remove extra dimension
        #     sdf[ijk[:, 0], ijk[:, 1], ijk[:, 2]] = pred_values  
        pred = self.test_fm_steps(all_inputs_vdb, model, n_steps)
        pred_ijk = pred.grid.ijk.jdata.cpu().detach().numpy()
        vector = all_inputs_vdb.jdata[:, 1:4].cpu().detach().numpy()  
        pred_ijk = (pred_ijk)*self.upsample_factor + (vector*(self.upsample_factor//2)).astype(int)
        pred_values = pred.jdata.detach().cpu().numpy().squeeze()  # Remove extra dimension
        sdf[pred_ijk[:, 0], pred_ijk[:, 1], pred_ijk[:, 2]] = pred_values
        
        sdf_mask = np.abs(sdf) < 100
        if actual_sdf is not None:
            # error between sdfs
            actual_values = actual_sdf[pred_ijk[:, 0], pred_ijk[:, 1], pred_ijk[:, 2]]
            error = np.abs(actual_values - pred_values)
            l1_error = np.mean(error)
            mean_squared_error = np.mean(error**2)
            
        # create a fvdb tensor from the sdf
        up_grid = fvdb.gridbatch_from_ijk(
                fvdb.JaggedTensor(torch.tensor(np.array(np.where(sdf_mask)).T)),
                voxel_sizes=(1/(upsampled_sdf_size-1)),
                origins=torch.tensor([0, 0, 0])
            )
        up_ijk = up_grid.ijk.jdata.cpu().detach().numpy()
        up_values = sdf[up_ijk[:, 0], up_ijk[:, 1], up_ijk[:, 2]]
        up_tensor = fvnn.VDBTensor(up_grid,
                                    up_grid.jagged_like(torch.tensor(up_values)))
        if actual_sdf is not None:
            # return up_tensor, error, mean_squared_error
            return up_tensor, l1_error, mean_squared_error
        else:
            # return up_tensor without error
            return up_tensor, None, None

    def save_predictions(self):
        # predictions_dir
        save_dir = os.path.join(self.save_predictions_dir, self.model_name)
        os.makedirs(save_dir, exist_ok=True)

        # load best model
        model_path = os.path.join(self.save_model_dir, f"{self.model_name}.pth")
        model = torch.load(model_path)
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model.to(device)
        model.eval()
        l1_errors = []
        mean_squared_errors = []
        names = []

        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc='Testing'):
                obj_names, vdb_input, new_ijkss, new_featuress, actual_sdfs = batch
                vdb_inputs = fvdb.jcat(vdb_input)
                new_ijks = new_ijkss[0]
                new_features = new_featuress[0]
                actual_sdf = actual_sdfs[0]
                # vdb_outputs = fvdb.jcat(vdb_output)

                up_tensor, l1_error, mean_squared_error = self.predictions_fm_steps(vdb_inputs, 
                                                      new_features, 
                                                      new_ijks, 
                                                      model, 
                                                      n_steps=10,
                                                      actual_sdf=actual_sdf)

                names.append(obj_names[0])
                l1_errors.append(l1_error)
                mean_squared_errors.append(mean_squared_error)

                # save the predictions
                file_names = [name.split('.')[0] for name in obj_names]
                output_file = os.path.join(save_dir, f'{file_names[0]}.nvdb')
                fvdb.save(output_file, up_tensor.grid, up_tensor.data, compressed=True)
                print(f"Saved predictions for {file_names[0]} to {output_file}")
                
        # log the errors
        df_error = pd.DataFrame({
            'object_name': names,
            'l1_error': l1_errors,
            'mean_squared_error': mean_squared_errors
        })
        df_error_describe = df_error.describe().reset_index()
        self.logger.log({'data/sdf_eval': wandb.Table(dataframe=df_error)})
        self.logger.log({'stats/sdf_eval': wandb.Table(dataframe=df_error_describe)})
        print(df_error_describe)

In [16]:
import torch
import os
import psutil

def set_my_gpu_priority():
    """Give maximum GPU priority to current user/process"""
    
    # Set high CPU priority for this process
    try:
        p = psutil.Process()
        p.nice(-20)  # Highest priority (-20 to 19, lower = higher priority)
        print(f"Process priority set to: {p.nice()}")
    except PermissionError:
        print("Run with sudo for highest priority, or use: sudo nice -n -20 python your_script.py")
    
    # GPU exclusive access
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
        # Set CUDA to use maximum performance
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.enabled = True
        
        # Set memory allocation strategy
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
        
        device = torch.cuda.current_device()
        print(f"Reserved GPU: {torch.cuda.get_device_name(device)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(device).total_memory / 1e9:.2f} GB")
        
        return torch.device('cuda')
    else:
        return torch.device('cpu')

# Call this at the start of your training
device = set_my_gpu_priority()

AccessDenied: (pid=4043699)

In [ ]:
st.set_reproducibility(is_reproducible=config['reproducibility']['is_reproducible'],
                           seed=config['reproducibility']['seed'])
    

## Model Simple CNN
vector_dim = 3
t_dim = 1
# CNN_vanilla_without_transpose, FVDBUNetBase
model = unetModels.FVDBUNetBase(
    in_channels=config['training']['in_channels'] + vector_dim + t_dim,
    out_channels=config['training']['out_channels'])
        
## optimizer
optimizer = torch.optim.Adam(model.parameters(), 
                            lr=config['training']['lr'], 
                            # weight_decay=1e-5
                            )
        
## training
# import threading

# next_dataset = None
# lock = threading.Lock()

# def prepare_next(epoch):
#     start = time.time()
#     global next_dataset
#     (train_dataloader, 
#     val_dataloader, 
#     test_dataloader) = ABCDataLoader(
#                                         input_dir=input_dir,
#                                         config=config,
#                                         # n_samples=20
#                                     ).get(names_set=names_set)
#     end = time.time()
#     print(f"Time taken to prepare dataset for epoch {epoch}: {end - start} seconds")
#     with lock:
#         next_dataset = (train_dataloader, val_dataloader, test_dataloader)

# kick off first dataset
# prepare_next(0)

trainer = ModelTrainer(
                        model_name=config['training']['model_name'],
                        model=model,
                        num_epochs=40,
                        train_loader=train_dataloader,
                        val_loader=val_dataloader,
                        test_loader=test_dataloader,
                        upsample_factor=config['data']['upsample_factor'],
                        input_size=config['data']['input_size'],
                        # pos_enc_dim=config['training']['positional_encoding'],
                        optimizer=optimizer,
                        loss_fn_name=config['training']['loss_function'],
                        # loss_weights=config['training']['loss_weights'],
                        is_save_model=config['training']['save_model'],
                        save_model_dir=config['training']['save_model_dir'],
                        save_predictions_dir=config['training']['save_predictions_dir'],
                        logger=logger
                    )

trainer.train()

## Evaluation
# evaluator = ABC_eval.Evaluator(
#                         model_name=config['training']['model_name'],
#                         # pos_enc_dim=config['training']['positional_encoding'],
#                         test_loader=test_dataloader,
#                         # upsampling_level=config['eval']['upsampling_level'],
#                         abc_dir=config['eval']['eval_dir'],
#                         save_model_dir=config['training']['save_model_dir'],
#                         save_predictions_dir=config['training']['save_predictions_dir'],
#                         n_job=config['eval']['eval_job'],
#                         eval_discription=config['eval']['eval_discription'],
#                         logger=logger
#                     )
# evaluator.evaluate()

Setting reproducibility with seed: 42


Epoch 1/40:   4%|▍         | 9/201 [00:40<13:36,  4.25s/it]

Setting reproducibility with seed: 42


Epoch 1/40: 100%|██████████| 201/201 [05:32<00:00,  1.66s/it]


Epoch 1/40, Loss: 0.2611


Validation: 100%|██████████| 67/67 [06:22<00:00,  5.71s/it]


Validation Loss: 0.1333
Model saved to /data/workspaces/spanwar/results/ssu/save_models/51_with_completely_random_vector_plus_new_data_processing.pth


...

 27%|██▋       | 1144/4280 [00:40<01:37, 32.30it/s]/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Epoch 3/40: 100%|██████████| 201/201 [05:46<00:00,  1.72s/it]


Epoch 3/40, Loss: 0.1564


Validation: 100%|██████████| 67/67 [04:47<00:00,  4.28s/it]


Validation Loss: 0.0937
Model saved to /data/workspaces/spanwar/results/ssu/save_models/51_with_completely_random_vector_plus_new_data_processing.pth


Epoch 4/40:  92%|█████████▏| 185/201 [09:18<01:00,  3.75s/it]

Setting reproducibility with seed: 42


Epoch 1/40: 100%|██████████| 201/201 [00:44<00:00,  4.54it/s]


Epoch 1/40, Loss: 0.2940


Validation: 100%|██████████| 67/67 [00:26<00:00,  2.49it/s]



Validation Loss: 0.1550
Model saved to /data/workspaces/spanwar/results/ssu/save_models/51_with_completely_random_cubic_vector.pth


...

100%|██████████| 4280/4280 [02:45<00:00, 25.78it/s]

Epoch 12/40: 100%|██████████| 201/201 [00:38<00:00,  5.21it/s]


Epoch 12/40, Loss: 0.0660


Validation: 100%|██████████| 67/67 [00:25<00:00,  2.63it/s]


Validation Loss: 0.0270


Epoch 13/40:  12%|█▏        | 25/201 [00:06<00:31,  5.56it/s]

In [ ]:
# from multiprocessing import Process
# def background_job():
#     for _ in range(15):
#         print(f"[Worker PID={os.getpid()}] Preparing data...")
#         run_data_processing(train_set + val_set)
#         time.sleep(5)

# # Start process
# worker = Process(target=background_job, daemon=True)
# worker.start()

[Worker PID=1244362] Preparing data...



  0%|          | 0/4280 [00:00<?, ?it/s]/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/joblib/parallel.py:1378: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = self._backend.configure(
/user/spanwar/home/.conda/envs/fvdb_ponq/lib/python3.11/site-packages/joblib/parallel.py:1378: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = self._backend.configure(
  1%|          | 30/4280 [01:20<6:11:10,  5.24s/it] 

In [19]:
worker.kill()

# Rough

In [8]:
def get_all_shifted_positions(vdb_tensor, size, upsample_factor):
        m3g = torch.tensor(mt.mesh_grid(upsample_factor+1), device=vdb_tensor.device) - (upsample_factor//2)

        new_ijks = []
        new_features = []
        for mg in m3g:
            ijk = vdb_tensor.grid.ijk.jdata
            ijk = (upsample_factor * ijk + mg).view(-1, 3)
            ijk = np.clip(ijk.cpu().detach().numpy(), 0, (size-1)*upsample_factor)
            ijk_vector = ijk - (vdb_tensor.grid.ijk.jdata.cpu().detach().numpy() * upsample_factor)
            ijk_vector = ijk_vector / (upsample_factor // 2)  # Normalize to values between -1 and 1
            ijk_vector = torch.tensor(ijk_vector, dtype=torch.float32, device=vdb_tensor.device)

            new_features.append(torch.cat([vdb_tensor.data.jdata.unsqueeze(1), ijk_vector], axis=-1))
            new_ijks.append(torch.tensor(ijk, dtype=torch.int, device=vdb_tensor.device))
        return new_features, new_ijks

In [42]:
import time
import torch

def get_sdf_sampled_data(obj_name):
    _dir_gt = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs'

    file_dir_gt = os.path.join(_dir_gt, obj_name.split('.')[0])
    # find the .obj file in the directory by walking through the directory
    obj_files = [f for f in os.listdir(file_dir_gt) if f.endswith('.obj')]
    file_gt = os.path.join(file_dir_gt, obj_files[0])
    
    # get sdf from the point
    gt_mesh = trimesh.load(file_gt)
    gt_mesh.vertices = gt_normalize_vertices(gt_mesh.vertices)

    gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    # read hdf5 file
    filename = obj_name.split('.')[0]
    h5_file = h5py.File(os.path.join(gt_large, f'{filename}.hdf5'), 'r')

    grid_n = 33
    sdf_32 = h5_file['32_sdf'][:]
    sdf_64 = h5_file['64_sdf'][:]
    mask_sdf_32 = mt.make_mask_close(sdf_32, (grid_n-1)*2)
    grid = mt.mesh_grid(grid_n, normalize=False)
    mask_sdf_32 = mask_sdf_32.flatten()
    grid_o = grid[mask_sdf_32]
    m3g = mt.mesh_grid(3)-1
    # m3g = np.array([0, 0, 0])*2

    # random select a random direction for each voxel   
    
    # print(f'grid shape: {grid.shape}')
    num_elements = grid_o.shape[0]
    random_indices = torch.randint(0, m3g.shape[0], (num_elements,))
    # # random number between 0 to 1
    rand_values = np.random.rand(num_elements, 3)

    # grid = (4*grid[:, None, :] + m3g[None, :, :]).reshape(-1, 3)
    # print(grid)
    grid = grid_o + m3g[random_indices] * rand_values/2
    # grid = grid_o
    m3g = np.array(mt.mesh_grid(3), dtype=np.float32)
    m3g[1] = [-1.0, -1.0, -0.5]
    print(m3g[1])
    grid = grid_o + m3g[1]/2
    # grid_val = m3g[random_indices] * rand_values
    # print(m3g[random_indices][:10])
    
    # print(grid_val)
    # clip
    # grid = np.clip(grid, 0, 4*(grid_n-1))

    grid = np.clip(grid, 0, grid_n-1)
    # print(rand_values[:10])
    # print(m3g[random_indices][:10])
    rand_values = (grid - grid_o)*2
    # print(rand_values[:10])
    print(m3g[1])
    # print(grid_o)

    # unique grid points
    # grid = np.unique(grid, axis=0)
    print(grid)
    grid_n_i = grid_n
    grid_n = grid/(grid_n-1)-0.5

    
    sub_start = time.time()
    sdf_val = igl.signed_distance(grid_n, gt_mesh.vertices, gt_mesh.faces)[0]
    sdf = np.full((grid_n_i, grid_n_i, grid_n_i), 1.0)
    sdf[grid_o[:, 0].astype(int), grid_o[:, 1].astype(int), grid_o[:, 2].astype(int)] = sdf_val
    sub_end = time.time()
    # print(f'SDF shape: {sdf.shape}')
    # print(sdf)
    # print(f'SDF shape: {sdf.shape}')
    print(f'SDF shape: {sdf.shape}')

    # read sdf
    # sdf_32 = h5_file['32_sdf'][:]
    sdf_128 = h5_file['128_sdf'][:]
     
    print(f'GT SDF shape: {sdf_128.shape}')
    # compare value of both sdf
    print(f'Max SDF value: {np.max(sdf)}')
    print(f'Min SDF value: {np.min(sdf)}')
    print(f'Max GT SDrandom_indicesF value: {np.max(sdf_128)}')
    print(f'Min GT SDF value: {np.min(sdf_128)}')
    # assert np.allclose(sdf, sdf_128), "SDF values do not match with GT SDF"
    # print(sdf)
    # print(10*'#')
    # print(sdf_128)
    # mt.plotSlice(sdf, 0.005)
    # plot 3d points
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D

    # Plot your grid points in 3D
    # fig = plt.figure(figsize=(10, 8))
    # ax = fig.add_subplot(111, projection='3d')

    # Assuming you have grid points from your function
    # ax.scatter(grid[:, 0], grid[:, 1], grid[:, 2], c='blue', s=20, alpha=0.6)
    # ax.set_xlabel('X')
    # ax.set_ylabel('Y')
    # ax.set_zlabel('Z')
    # ax.set_title('3D Grid Points')
    # plt.show()
    
    ijk = torch.tensor(grid_o, dtype=torch.int, device='cpu')

    fvdb_grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(ijk),
                                        voxel_sizes=(1/(33-1)),
                                   origins=torch.tensor([0, 0, 0], device='cpu')
                                   )
    ijk = fvdb_grid.ijk.jdata
    sdf_val = sdf[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    sdf_val_i = torch.tensor(sdf_val, dtype=torch.float32, device='cpu')
    voxel_vector = torch.tensor(rand_values, dtype=torch.float32, device='cpu')
    print(sdf_val_i.shape, voxel_vector.shape)
    sdf_val_cat = torch.cat([sdf_val_i[:,None], voxel_vector], dim=-1)
    input_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(torch.tensor(sdf_val_cat)))
    vdb_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(torch.tensor(sdf_val)))
    v, f, _ = vdb_tensor.grid.marching_cubes(vdb_tensor.data)
    # plot(v.jdata.numpy(), f.jdata.numpy())

    # sdf_new = np.full((65, 65, 65), 100.0)
    # sdf_new[ijk[:, 0], ijk[:, 1], ijk[:, 2]] = sdf[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    # v, f, _, _ = measure.marching_cubes(sdf_new, level=0.0)
    # plot(v, f)
    # sort grid_o
    # grid_o = grid_o[np.lexsort((grid_o[:, 2], grid_o[:, 1], grid_o[:, 0]))]
    # ijk = ijk[np.lexsort((ijk[:, 2], ijk[:, 1], ijk[:, 0]))]
    print(input_tensor.data.jdata[:3])
    print(grid_o[:3])
    print(ijk[:3])
    ijk_new = ijk*4 + m3g[1]*2
    ijk_new = torch.tensor(ijk_new, dtype=torch.int, device='cpu')
    print('new ijk orgi', ijk_new)
    # ijk_new = (grid * 4).astype(int)
    # print('new ijk', ijk_new)
    check_sdf = sdf_128[ijk_new[:, 0], ijk_new[:, 1], ijk_new[:, 2]]
    print(np.mean((abs(check_sdf - sdf_val)*65)))
    print(m3g[1]/2)
    print(check_sdf*65)
    print(sdf_val*65)
    features, ijks = get_all_shifted_positions(vdb_tensor, grid_n_i, 4)
    print(ijks[1])
    print(features[1])
    print(sdf_128[ijks[1][:, 0], ijks[1][:, 1], ijks[1][:, 2]]*65)
    # assert np.allclose(check_sdf*65, sdf_val*65), "SDF values do not match with GT SDF"
    # v, f, _, _ = measure.marching_cubes(sdf, level=0.0)
    # plot(v, f)
    # v, f, _, _ = measure.marching_cubes(sdf_128, level=0.0)
    # plot(v, f)
    # return sdf
get_sdf_sampled_data('00000020.obj')

[-1.  -1.  -0.5]
[-1.  -1.  -0.5]
[[11.5   2.5  14.75]
 [11.5   2.5  15.75]
 [11.5   2.5  16.75]
 ...
 [19.5  26.5  17.75]
 [19.5  26.5  18.75]
 [19.5  26.5  19.75]]
SDF shape: (33, 33, 33)
GT SDF shape: (129, 129, 129)
Max SDF value: 1.0
Min SDF value: -0.048459206203273465
Max GT SDrandom_indicesF value: 0.5854235887527466
Min GT SDF value: -0.04958410933613777
torch.Size([2328]) torch.Size([2328, 3])
tensor([[ 0.0670, -1.0000, -1.0000, -0.5000],
        [ 0.0652, -1.0000, -1.0000, -0.5000],
        [ 0.0608, -1.0000, -1.0000, -0.5000]])
[[12  3 15]
 [12  3 16]
 [12  3 17]]
tensor([[12,  3, 15],
        [12,  4, 14],
        [12,  4, 15]], dtype=torch.int32)
new ijk orgi tensor([[ 46,  10,  59],
        [ 46,  14,  55],
        [ 46,  14,  59],
        ...,
        [ 78, 106,  71],
        [ 78, 106,  75],
        [ 78, 106,  79]], dtype=torch.int32)
7.027357991385476e-06
[-0.5  -0.5  -0.25]
[4.3546543 4.2408624 3.9527478 ... 1.9205931 1.9205931 1.9205931]
[4.35465375 4.24086181 3.95

/tmp/ipykernel_771789/4236475095.py:122: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(torch.tensor(sdf_val_cat)))
/tmp/ipykernel_771789/4236475095.py:137: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  ijk_new = ijk*4 + m3g[1]*2
/tmp/ipykernel_771789/4236475095.py:138: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ijk_new = torch.tensor(ijk_new, dtype=torch.int, device='cpu')


In [37]:
(mt.mesh_grid(5)-2)/2

array([[-1. , -1. , -1. ],
       [-1. , -1. , -0.5],
       [-1. , -1. ,  0. ],
       [-1. , -1. ,  0.5],
       [-1. , -1. ,  1. ],
       [-1. , -0.5, -1. ],
       [-1. , -0.5, -0.5],
       [-1. , -0.5,  0. ],
       [-1. , -0.5,  0.5],
       [-1. , -0.5,  1. ],
       [-1. ,  0. , -1. ],
       [-1. ,  0. , -0.5],
       [-1. ,  0. ,  0. ],
       [-1. ,  0. ,  0.5],
       [-1. ,  0. ,  1. ],
       [-1. ,  0.5, -1. ],
       [-1. ,  0.5, -0.5],
       [-1. ,  0.5,  0. ],
       [-1. ,  0.5,  0.5],
       [-1. ,  0.5,  1. ],
       [-1. ,  1. , -1. ],
       [-1. ,  1. , -0.5],
       [-1. ,  1. ,  0. ],
       [-1. ,  1. ,  0.5],
       [-1. ,  1. ,  1. ],
       [-0.5, -1. , -1. ],
       [-0.5, -1. , -0.5],
       [-0.5, -1. ,  0. ],
       [-0.5, -1. ,  0.5],
       [-0.5, -1. ,  1. ],
       [-0.5, -0.5, -1. ],
       [-0.5, -0.5, -0.5],
       [-0.5, -0.5,  0. ],
       [-0.5, -0.5,  0.5],
       [-0.5, -0.5,  1. ],
       [-0.5,  0. , -1. ],
       [-0.5,  0. , -0.5],
 

In [61]:
os.listdir('/data/workspaces/spanwar/results/ssu/save_models')

['SSU_PONQ_DATA_UPSAMPLER_Unet.pth',
 'SSU_PONQ_DATA_UPSAMPLER.pth',
 '25_rerun_24_SSU_PONQ_DATA_UPSAMPLER_FM_trilinear_Unet_v2.pth',
 '26_rerun_24_SSU_PONQ_DATA_UPSAMPLER_FM_trilinear_Unet_v2.pth',
 '24_SSU_PONQ_DATA_UPSAMPLER_FM_trilinear_Unet_v2.pth',
 '27_Unet_Upsampler_Manual_NMC_data.pth',
 '28_Unet_Upsampler_Manual_NMC_data_l1.pth',
 '29_Unet_Upsampler_Manual_NMC_data_l1.pth',
 '30_Unet_Upsampler_Manual_NMC_data_l1.pth',
 '31_Simple_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '32_Simple_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '33_Simple_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '34_Simple_CNN_Upsampler_Manual_NMC_data_huber_loss.pth',
 '35_rerun_33_Simple_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '36_rerun_35_Simple_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '37_residual_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '38_rerun_37_residual_CNN_Upsampler_Manual_NMC_data_l1.pth',
 '39_residual_CNN_Upsampler_Manual_NMC_data_l1_&_sign_loss.pth',
 '40_rerun_31_CNN_Upsampler_Manual_NMC_data_l1_&_sign_lo

In [62]:
def predict_sdf(input):
    model_name = '51_with_completely_random_vector_plus_new_data_processing.pth'
    model_dir = '/data/workspaces/spanwar/results/ssu/save_models'
    model_path = os.path.join(model_dir, model_name)
    model = torch.load(model_path)
    model.eval()
    with torch.no_grad():
        return model(input)

In [14]:
import open3d as o3d
def compute_sdf_open3d(vertices, faces, query_points):
    """
    Open3D is 3-5x faster than IGL for SDF computation
    """
    # Create mesh
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.triangles = o3d.utility.Vector3iVector(faces)
    
    # Create scene for ray casting
    scene = o3d.t.geometry.RaycastingScene()
    mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
    scene.add_triangles(mesh_t)
    
    # Query points
    query_points_t = o3d.core.Tensor(query_points, dtype=o3d.core.Dtype.Float32)
    
    # Compute signed distances
    sdf = scene.compute_signed_distance(query_points_t)
    
    return sdf.numpy()


In [24]:
import time
import torch

def get_sdf_sampled_data(obj_name):
    _dir_gt = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs'

    file_dir_gt = os.path.join(_dir_gt, obj_name.split('.')[0])
    # find the .obj file in the directory by walking through the directory
    obj_files = [f for f in os.listdir(file_dir_gt) if f.endswith('.obj')]
    file_gt = os.path.join(file_dir_gt, obj_files[0])
    
    # get sdf from the point
    gt_mesh = trimesh.load(file_gt)
    gt_mesh.vertices = gt_normalize_vertices(gt_mesh.vertices)

    gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    # read hdf5 file
    filename = obj_name.split('.')[0]
    h5_file = h5py.File(os.path.join(gt_large, f'{filename}.hdf5'), 'r')

    grid_n = 33
    sdf_32 = h5_file['32_sdf'][:]
    sdf_64 = h5_file['64_sdf'][:]
    mask_sdf_32 = mt.make_mask_close(sdf_32, ((grid_n-1)*2)+1)
    grid = mt.mesh_grid(grid_n, normalize=False)
    mask_sdf_32 = mask_sdf_32.flatten()
    grid_o = grid[mask_sdf_32]
    m3g = mt.mesh_grid(5)-2
    m3g = m3g/2

    grids = []
    grids_norm = []
    voxel_vectors = []
    for mg in m3g:
        grid = grid_o + mg/2
        grid = np.clip(grid, 0, grid_n-1)        
        grids.append(grid)

        grid_norm = grid/(grid_n-1)-0.5
        grids_norm.append(grid_norm)
        voxel_vectors.append((grid - grid_o)*2)

    sdf_values = []
    for grid_norm in grids_norm:
        # sdf_val = igl.signed_distance(grid_norm, gt_mesh.vertices, gt_mesh.faces)[0]
        sdf_val = compute_sdf_open3d(gt_mesh.vertices, gt_mesh.faces, grid_norm)
        sdf = np.full((grid_n, grid_n, grid_n), 1.0)
        sdf[grid_o[:, 0].astype(int), grid_o[:, 1].astype(int), grid_o[:, 2].astype(int)] = sdf_val
        sdf_values.append(sdf)
        # v, f, _, _ = measure.marching_cubes(sdf, level=0.0)
        # plot(v, f)
        # break
    
    sdf_128 = h5_file['128_sdf'][:]
    sdf_32 = h5_file['32_sdf'][:]

    ijk = torch.tensor(grid_o, dtype=torch.int, device='cpu')

    fvdb_grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(ijk),
                                        voxel_sizes=(1/(33-1)),
                                   origins=torch.tensor([0, 0, 0], device='cpu')
                                   )
    ijk = fvdb_grid.ijk.jdata
    vdb_tensor = fvnn.VDBTensor(fvdb_grid, fvdb_grid.jagged_like(torch.tensor(sdf_32[ijk[:, 0], ijk[:, 1], ijk[:, 2]])))
    features, ijks = get_all_shifted_positions(vdb_tensor, grid_n, 4)
    query_sdfs = []
    mean_errors = []
    sdf = np.full((129, 129, 129), 0.01)
    for _ijk, sdf_value in zip(ijks, sdf_values):
        # assert _ijk == (4*grid)
        sdf[_ijk[:, 0], _ijk[:, 1], _ijk[:, 2]] = sdf_value[ijk[:, 0], ijk[:, 1], ijk[:, 2]]*65
        igl_value = sdf_value[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        actual_value = sdf_128[_ijk[:, 0], _ijk[:, 1], _ijk[:, 2]]
        # print(igl_value[:10]*65)
        # print(actual_value[:10]*65)
        mean_error = np.mean(np.abs(igl_value - actual_value))*65
        mean_errors.append(mean_error)
        # assert np.allclose(igl_value, actual_value), "SDF values do not match with GT SDF"
        # sdf = np.full((33, 33, 33), 100.0)
        # sdf[ijk[:, 0], ijk[:, 1], ijk[:, 2]] = sdf_value[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    print(f'Mean error of IGL SDF: {np.mean(mean_errors)}')
    sdf_32_igl = np.full((33, 33, 33), 10.0)
    sdf_32_igl[ijk[:, 0], ijk[:, 1], ijk[:, 2]] = sdf_value[ijk[:, 0], ijk[:, 1], ijk[:, 2]]*65
    v, f, _, _ = measure.marching_cubes(sdf_32_igl, level=0.0)
    plot(v, f)
    
    v, f, _, _ = measure.marching_cubes(sdf, level=0.0)
    plot(v, f)

    v, f, _, _ = measure.marching_cubes(sdf_128, level=0.0)
    plot(v, f)

    nv, nf, _ = vdb_tensor.grid.marching_cubes(vdb_tensor.data)
    plot(nv.jdata.numpy(), nf.jdata.numpy())

get_sdf_sampled_data('00000020.obj')

Mean error of IGL SDF: 1.7182048851408344e-05


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(15.5, 15.…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(64.0, 64.…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(64.0, 64.…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.50…

In [127]:
import os
import sys
import igl
import time
import h5py
import torch
import joblib
import random
import trimesh
import numpy as np
import fvdb
from tqdm import tqdm
import fvdb.nn as fvnn
import open3d as o3d
import subprocess

sys.path.append('../src')
from utils import mesh_tools as mt


def compute_sdf_open3d(vertices, faces, query_points):
    """
    Open3D is 3-5x faster than IGL for SDF computation
    """
    # Create mesh
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.triangles = o3d.utility.Vector3iVector(faces)
    
    # Create scene for ray casting
    scene = o3d.t.geometry.RaycastingScene()
    mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
    scene.add_triangles(mesh_t)
    
    if type(query_points) is list:
        sdf_list = []
        for q_points in query_points:
            q_points_t = o3d.core.Tensor(q_points, dtype=o3d.core.Dtype.Float32)
            sdf = scene.compute_signed_distance(q_points_t)
            sdf_list.append(torch.tensor(sdf.numpy(), dtype=torch.float32, device='cpu'))

    # Query points
    query_points_t = o3d.core.Tensor(query_points, dtype=o3d.core.Dtype.Float32)
    
    # Compute signed distances
    sdf = scene.compute_signed_distance(query_points_t)

    return torch.tensor(sdf.numpy(), dtype=torch.float32, device='cpu')


def gt_normalize_vertices(vertices):
    """
    Normalize vertices and optionally return normalization parameters
    """
    #normalize diagonal=1
    x_max = np.max(vertices[:,0])
    y_max = np.max(vertices[:,1])
    z_max = np.max(vertices[:,2])
    x_min = np.min(vertices[:,0])
    y_min = np.min(vertices[:,1])
    z_min = np.min(vertices[:,2])
    x_mid = (x_max+x_min)/2
    y_mid = (y_max+y_min)/2
    z_mid = (z_max+z_min)/2
    x_scale = x_max - x_min
    y_scale = y_max - y_min
    z_scale = z_max - z_min
    scale = np.sqrt(x_scale*x_scale + y_scale*y_scale + z_scale*z_scale)
    
    vertices[:,0] = (vertices[:,0]-x_mid)/scale
    vertices[:,1] = (vertices[:,1]-y_mid)/scale
    vertices[:,2] = (vertices[:,2]-z_mid)/scale

    return vertices

def random_rotation_matrix(gt_mesh):
    # Generate random rotation matrix
    angles = np.random.uniform(0, 2*np.pi, 3)  # Random angles for x, y, z
    
    # Rotation matrices for each axis
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(angles[0]), -np.sin(angles[0])],
                   [0, np.sin(angles[0]), np.cos(angles[0])]])
    
    Ry = np.array([[np.cos(angles[1]), 0, np.sin(angles[1])],
                   [0, 1, 0],
                   [-np.sin(angles[1]), 0, np.cos(angles[1])]])
    
    Rz = np.array([[np.cos(angles[2]), -np.sin(angles[2]), 0],
                   [np.sin(angles[2]), np.cos(angles[2]), 0],
                   [0, 0, 1]])
    
    R = Rz @ Ry @ Rx
    gt_mesh.vertices = gt_mesh.vertices @ R.T
    return gt_mesh

def scaled_sdf(threshold, sdf_arr: np.array):
    '''scales the SDF array by the threshold value'''
    return (threshold-1)*sdf_arr[:, None]

def fetch_numpy_values(grid: fvdb.GridBatch, arr: np.array, size:int):
        '''fetches values from a numpy array based on the ijk indices in the grid'''
        ijk = grid.ijk.jdata.cpu().detach().numpy()
        
        if max(ijk[:, 0]) >= arr.shape[0] or max(ijk[:, 1]) >= arr.shape[1] or max(ijk[:, 2]) >= arr.shape[2]:
            # If indices are out of bounds, we can add the maximum value to the indices
            ijk = np.clip(ijk, 0, np.array(arr.shape) - 1)
            # print(f"Indices out of bounds. Clipping to max shape: {arr.shape}")
        
        values = arr[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        return torch.tensor(values, dtype=torch.float32, device=grid.device)

def fetch_numpy_values_shifted(ijk, arr: np.array):
    '''fetches values from a numpy array based on the ijk indices in the grid'''
    ijk = ijk.cpu().detach().numpy()
    values = arr[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    device = 'cpu'
    return torch.tensor(values, dtype=torch.float32, device=device)

def path_information(path_name):
    paths = {
        'fvdb_saved_dir': '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random',
        'sdf_gt_large': '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large',
        'gt_objs': '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/abc_dataset_objs'
    }
    return paths.get(path_name, None)

class DataProcessing:
    def __init__(self,
                 names_set, 
                 input_size, 
                 threshold, 
                 random_direction_type,
                 fvdb_saved_dir,
                 sdf_gt_large,
                 gt_objs):
        self.names_set = names_set
        self.input_size = input_size
        self.threshold = threshold
        self.random_direction_type = random_direction_type
        self.fvdb_saved_dir = fvdb_saved_dir
        self.sdf_gt_large = sdf_gt_large
        self.gt_objs = gt_objs

    @staticmethod
    def read_sdf_file_with_offsets(name):
        with open(name, 'rb') as fp:
            # Read header
            line = fp.readline().strip()
            if not line.startswith(b'#sdf'):
                raise IOError('Not a sdf file')
            
            # Read dimensions
            dims = list(map(int, fp.readline().strip().split(b' ')[1:]))
            
            # Skip "data" line
            fp.readline()
            
            # Read data point by point (1 float + 3 ints per grid point)
            total_elements = dims[0] * dims[1] * dims[2]
            sdf_data = np.zeros(dims, dtype=np.float32)
            offset_data = np.zeros(dims + [3], dtype=np.int32)
            
            for i in range(dims[0]):
                for j in range(dims[1]):
                    for k in range(dims[2]):
                        # Read phi value (float)
                        phi_bytes = fp.read(4)
                        phi_value = np.frombuffer(phi_bytes, dtype=np.float32)[0]
                        sdf_data[i, j, k] = phi_value
                        
                        # Read 3 offset values (ints)
                        offset_bytes = fp.read(12)  # 3 ints * 4 bytes
                        offsets = np.frombuffer(offset_bytes, dtype=np.int32)
                        offset_data[i, j, k] = offsets
            
            return sdf_data, offset_data

    def get_processed_sdf_data(self, obj_name,
                               threshold=65,
                               grid_size=33,
                               random_direction_type='nonUniform'):
        output_path = self.fvdb_saved_dir
        file_name = obj_name.split('.')[0]

        _dir_gt = self.gt_objs
        obj_file = os.path.join(_dir_gt, file_name, 'model.obj')
        # file_dir_gt = os.path.join(_dir_gt, obj_name.split('.')[0])
        # obj_files = [f for f in os.listdir(file_dir_gt) if f.endswith('.obj')]
        
        # all paths
        # file_gt = os.path.join(file_dir_gt, obj_files[0])
        file_gt = obj_file
        sdf_file_path = os.path.join(output_path, f'{file_name}.sdf')
        input_vdb_path = os.path.join(output_path, 
                                    f'{file_name}_input.nvdb')
        output_vdb_path = os.path.join(output_path, 
                                    f'{file_name}_output.nvdb')

        # factor = random.randint(1, 5)*2
        factor = 8

        # run c++ binary file
        # run_cmd = f"/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/src/data_processing/run {file_gt} {sdf_file_path} {factor}"
        # os.system(run_cmd)
        run_cmd = [
                "/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/src/data_processing/run",
                file_gt,
                sdf_file_path,
                str(factor)
            ]
        # print("Running command:", ' '.join(run_cmd))
        proc = subprocess.run(run_cmd, 
                        check=True,  # Raises exception on non-zero exit
                        capture_output=True,  # Capture stdout/stderr
                        text=True,  # Return strings instead of bytes
                        # timeout=300
                        )
        # read sdf file
        gt_large = self.sdf_gt_large
        with h5py.File(os.path.join(gt_large, f'{file_name}.hdf5'), 'r') as h5_file:
            sdf_32 = h5_file['32_sdf'][:]
            sdf_128 = h5_file['256_sdf'][:]
        sdf_data, offset_data = self.read_sdf_file_with_offsets(sdf_file_path)
        offset_data = offset_data/(factor//2)
        # print(factor, file_name, (sdf_data<0).sum(), sdf_32.shape, offset_data.shape)
        # input and output sdfs preparation
        # print((np.abs(sdf_data) < 3/(2*32)).sum())
        threshold = (factor//2) * (grid_size-1) + 1
        print(f'Using threshold: {threshold}')
        mask = mt.make_mask_close(sdf_data, threshold)
        # print(mask.sum())

        #  create a grid of the size without nomalize actual shape
        ijk_mesh_grid = mt.mesh_grid(grid_size)
        ijk_mesh_grid = ijk_mesh_grid.reshape(grid_size, grid_size, grid_size, 3)

        ijk = torch.tensor(ijk_mesh_grid[mask], 
                            dtype=torch.int, 
                            device='cpu')
        grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(ijk), 
                                        voxel_sizes=(1/(grid_size-1)), 
                                        origins=torch.tensor([0, 0, 0], 
                                        device='cpu'))
        
        # scale and mask
        ijk = grid.ijk.jdata
        sdf_data = sdf_data[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        sdf_32 = sdf_32[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        sdf_data = scaled_sdf(threshold, sdf_data)
        sdf_32 = scaled_sdf(threshold, sdf_32)
        sdf_data = torch.tensor(sdf_data, dtype=torch.float32, device='cpu')
        sdf_32 = torch.tensor(sdf_32, dtype=torch.float32, device='cpu')
        offset_data = offset_data[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
        offset_data = torch.tensor(offset_data, dtype=torch.float32, device='cpu')

        actual_ijk = ijk*factor + offset_data*(factor//2)
        actual_ijk = actual_ijk.int()
        actual_sdf = fetch_numpy_values_shifted(actual_ijk, sdf_128)
        actual_sdf = scaled_sdf(threshold, actual_sdf)
        # print('actual ijk orgi', actual_ijk)
        # print(mask.shape, ijk.shape, sdf_data.shape, sdf_32.shape, offset_data.shape)
        # create VDBTensor
        shifted_vdb = fvnn.VDBTensor(grid, 
                                    grid.jagged_like(sdf_data))

        small_features = torch.cat([sdf_32, offset_data], dim=-1) 
        small_vdb = fvnn.VDBTensor(grid, 
                                    grid.jagged_like(small_features))
        # fvdb.save(input_vdb_path, small_vdb.grid, small_vdb.data, compressed=True)
        # fvdb.save(output_vdb_path, shifted_vdb.grid, shifted_vdb.data, compressed=True)
        return small_vdb, shifted_vdb, actual_sdf

    
    def run_data_processing(self):
        joblib.Parallel(n_jobs=-1)(joblib.delayed(self.get_processed_sdf_data)
                                   (name, 
                                    threshold=self.threshold,
                                    grid_size=self.input_size,
                                    random_direction_type=self.random_direction_type) 
                                    for name in tqdm(self.names_set))
        # for name in tqdm(self.names_set):
        #     self.get_processed_sdf_data(name, 
        #     threshold=self.threshold,
        #     grid_size=self.input_size,
        #     random_direction_type=self.random_direction_type) 
                                    

In [128]:
from data_processing import data_processing
config = read_config.read_yaml_config('/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/config/config_51_14082025_1100.yaml')

dataProcessor = DataProcessing(None, 
                                            input_size=config['data']['input_size'],
                                            threshold=config['data']['mask_threshold'],
                                            random_direction_type=config['data']['random_direction_type'],
                                            fvdb_saved_dir=config['data']['fvdb_saved_dir'],
                                            sdf_gt_large=config['data']['input_dir'],
                                            gt_objs=config['eval']['eval_dir'])

In [129]:
small_vdb, shifted_vdb, actual_sdf = dataProcessor.get_processed_sdf_data('00000020.obj')

Using threshold: 129


In [130]:
small_vdb.data.jdata, small_vdb.data.jdata.shape

(tensor([[ 2.0892,  1.0000,  0.2500, -0.5000],
         [ 1.7821,  0.0000, -0.7500,  0.2500],
         [ 1.7821,  0.7500, -0.2500,  0.7500],
         ...,
         [ 5.0459, -0.5000, -0.7500, -1.0000],
         [ 2.9604, -0.5000,  0.2500,  0.7500],
         [ 2.9501, -1.0000,  0.2500, -0.2500]]),
 torch.Size([1182, 4]))

In [120]:
shifted_vdb.data.jdata[:,0]

tensor([2.2123, 1.7821, 1.7821,  ..., 2.4262, 1.3309, 2.9519])

In [121]:
actual_sdf

tensor([[2.2123],
        [1.7821],
        [1.7821],
        ...,
        [2.4262],
        [1.3309],
        [2.9519]])

In [123]:
(shifted_vdb.data.jdata[:,0].unsqueeze(1)-actual_sdf).abs().sum()/len(actual_sdf)

tensor(4.1304e-05)

In [75]:
# shifted_vdb.data.jdata

In [76]:
# (shifted_vdb.data.jdata - shifted_vdb_a.data.jdata).abs().sum()/2292

In [77]:
v, f,_ = small_vdb.grid.marching_cubes(small_vdb.data.jdata[:,0])
plot(v.jdata.numpy(), f.jdata.numpy())
v, f, _ = shifted_vdb.grid.marching_cubes(shifted_vdb.data.jdata[:,0])
plot(v.jdata.numpy(), f.jdata.numpy())
# v, f, _ = shifted_vdb_a.grid.marching_cubes(shifted_vdb_a.data.jdata[:,0])
# plot(v.jdata.numpy(), f.jdata.numpy())

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.50…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4998081…

In [47]:
m3g = mt.mesh_grid(5)-2
m3g*0.8

array([[-1.6, -1.6, -1.6],
       [-1.6, -1.6, -0.8],
       [-1.6, -1.6,  0. ],
       [-1.6, -1.6,  0.8],
       [-1.6, -1.6,  1.6],
       [-1.6, -0.8, -1.6],
       [-1.6, -0.8, -0.8],
       [-1.6, -0.8,  0. ],
       [-1.6, -0.8,  0.8],
       [-1.6, -0.8,  1.6],
       [-1.6,  0. , -1.6],
       [-1.6,  0. , -0.8],
       [-1.6,  0. ,  0. ],
       [-1.6,  0. ,  0.8],
       [-1.6,  0. ,  1.6],
       [-1.6,  0.8, -1.6],
       [-1.6,  0.8, -0.8],
       [-1.6,  0.8,  0. ],
       [-1.6,  0.8,  0.8],
       [-1.6,  0.8,  1.6],
       [-1.6,  1.6, -1.6],
       [-1.6,  1.6, -0.8],
       [-1.6,  1.6,  0. ],
       [-1.6,  1.6,  0.8],
       [-1.6,  1.6,  1.6],
       [-0.8, -1.6, -1.6],
       [-0.8, -1.6, -0.8],
       [-0.8, -1.6,  0. ],
       [-0.8, -1.6,  0.8],
       [-0.8, -1.6,  1.6],
       [-0.8, -0.8, -1.6],
       [-0.8, -0.8, -0.8],
       [-0.8, -0.8,  0. ],
       [-0.8, -0.8,  0.8],
       [-0.8, -0.8,  1.6],
       [-0.8,  0. , -1.6],
       [-0.8,  0. , -0.8],
 

In [66]:
grid.ijk.jdata[:10], feature.jdata[:10]

(tensor([[46, 14, 54],
         [46, 14, 55],
         [46, 15, 54],
         [46, 15, 55],
         [47, 14, 54],
         [47, 14, 55],
         [47, 15, 54],
         [47, 15, 55],
         [46, 10, 62],
         [46, 10, 63]], dtype=torch.int32),
 tensor([2.8748, 2.8835, 2.8834, 2.8929, 2.8719, 2.8827, 2.8798, 2.8903, 2.9460,
         2.9656], dtype=torch.float64))

In [150]:
# read sdf hdf5 file
gt_large =  '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
h5_file = h5py.File(os.path.join(gt_large, f'00000027.hdf5'), 'r')
sdf_129 = h5_file['32_sdf'][:]
v,f,_,_ = measure.marching_cubes(sdf_129, level=0.0)
plot(v, f)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(16.0, 15.…

In [ ]:
pred_dir = '/data/workspaces/spanwar/preprocessed_data/ssu/51_complete_random'
grid, feature, _ = fvdb.load(os.path.join(pred_dir, '00000002_input.nvdb'), device='cpu')
print(feature.jdata)
feature.jdata = feature.jdata[:, 0].unsqueeze(1)
nv, nf, _ = grid.marching_cubes(feature)
plot(nv.jdata.numpy(), nf.jdata.numpy())

tensor([[ 2.9420, -0.3745,  0.9507,  0.7320],
        [ 2.8910, -0.5987,  0.1560,  0.1560],
        [ 2.8910, -0.0581,  0.8662,  0.6011],
        ...,
        [ 2.8910, -0.1797,  0.5781,  0.9580],
        [ 2.8910, -0.5031,  0.0829,  0.6200],
        [ 2.9106, -0.3402,  0.8559,  0.4347]])


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.50…

In [149]:
pred_dir = '/data/workspaces/spanwar/results/ssu/test_predictions/51_with_completely_random_vector_plus_new_data_processing'
grid, feature, _ = fvdb.load(os.path.join(pred_dir, '00000027.nvdb'), device='cpu')
nv, nf, _ = grid.marching_cubes(feature)
plot(nv.jdata.numpy(), nf.jdata.numpy())

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5037886…

In [141]:
import numpy as np

def read_sdf_file_with_offsets(name):
    """
    Read SDF file containing [phi(float), phi_org(float), offset_i(int), offset_j(int), offset_k(int)] per grid point
    Returns: (sdf_data, sdf_org_data, offset_data)
    """
    with open(name, 'rb') as fp:
        # Read header
        line = fp.readline().strip()
        if not line.startswith(b'#sdf'):
            raise IOError('Not a sdf file')
        
        # Read dimensions
        dims = list(map(int, fp.readline().strip().split(b' ')[1:]))
        
        # Skip "data" line
        fp.readline()
        
        # Read data point by point (2 floats + 3 ints per grid point)
        total_elements = dims[0] * dims[1] * dims[2]
        sdf_data = np.zeros(dims, dtype=np.float32)
        sdf_org_data = np.zeros(dims, dtype=np.float32)  # Add original SDF
        offset_data = np.zeros(dims + [3], dtype=np.int32)
        
        for i in range(dims[0]):
            for j in range(dims[1]):
                for k in range(dims[2]):
                    # Read phi value (float) - modified SDF
                    phi_bytes = fp.read(4)
                    phi_value = np.frombuffer(phi_bytes, dtype=np.float32)[0]
                    sdf_data[i, j, k] = phi_value
                    
                    # Read phi_org value (float) - original SDF
                    phi_org_bytes = fp.read(4)
                    phi_org_value = np.frombuffer(phi_org_bytes, dtype=np.float32)[0]
                    sdf_org_data[i, j, k] = phi_org_value
                    
                    # Read 3 offset values (ints)
                    offset_bytes = fp.read(12)  # 3 ints * 4 bytes
                    offsets = np.frombuffer(offset_bytes, dtype=np.int32)
                    offset_data[i, j, k] = offsets
        
        return sdf_data, sdf_org_data, offset_data

# Usage
sdf_data, sdf_org_data, offset_data = read_sdf_file_with_offsets('/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/run/data/test_model.sdf')

print(f"SDF shape: {sdf_data.shape}")
print(f"SDF original shape: {sdf_org_data.shape}")
print(f"Offset data shape: {offset_data.shape}")

# Access phi, phi_org and offset for grid point (i,j,k)
i, j, k = 10, 15, 20
phi_value = sdf_data[i, j, k]
phi_org_value = sdf_org_data[i, j, k]
offset_i, offset_j, offset_k = offset_data[i, j, k]
print(f"Grid point ({i},{j},{k}): phi={phi_value:.6f}, phi_org={phi_org_value:.6f}, offset=({offset_i}, {offset_j}, {offset_k})")

# Compare the two SDFs
difference = np.abs(sdf_data - sdf_org_data)
print(f"Max difference between phi and phi_org: {np.max(difference):.6f}")
print(f"Mean difference between phi and phi_org: {np.mean(difference):.6f}")
print(f"Number of points with difference > 0.001: {np.sum(difference > 0.001)}")

SDF shape: (33, 33, 33)
SDF original shape: (33, 33, 33)
Offset data shape: (33, 33, 33, 3)
Grid point (10,15,20): phi=3.093750, phi_org=3.093750, offset=(-4, -4, -3)
Max difference between phi and phi_org: 0.024981
Mean difference between phi and phi_org: 0.001290
Number of points with difference > 0.001: 3675


In [142]:
offset_data

array([[[[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]],

        [[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]],

        [[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]],

        ...,

        [[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]],

        [[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]],

        [[-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3],
         ...,
         [-4, -4, -3],
         [-4, -4, -3],
         [-4, -4, -3]]],


       [[[-4, -4, -3],
         [-4, -4, -3],
         [-4, 

In [143]:
sdf_data

array([[[3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        ...,
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375]],

       [[3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        ...,
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375]],

       [[3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
        [3.09375, 3.09375, 3.09375, ..., 3.09375, 3.09375, 3.09375],
    

In [144]:
sdf_data = sdf_org_data.copy()
# mask = (sdf_data < 1) & (sdf_data > -1)
mask = np.abs(sdf_data) < 3/(2*32)
mask.sum(), (sdf_data<0).sum()

(np.int64(2328), np.int64(422))

In [145]:
indexs = mt.mesh_grid(33)
sdf_test = np.full((33,33,33), 10.0)
offset = 0
x_idx = np.minimum(indexs[:,0]*8+offset, 256)
y_idx = np.minimum(indexs[:,1]*8+offset, 256)
z_idx = np.minimum(indexs[:,2]*8+offset, 256)
sdf_test[indexs[:,0], indexs[:,1], indexs[:,2]] = sdf_256[x_idx, y_idx, z_idx]
# sdf_test[indexs[:,0], indexs[:,1], indexs[:,2]] = sdf_32[indexs[:,0], indexs[:,1], indexs[:,2]]

mismatch = (sdf_test[mask] != sdf_data[mask])
mask_indices = np.argwhere(mask)
mismatch_indices = mask_indices[mismatch]
print(len(mismatch_indices))
# assert np.allclose(sdf_test[mask], sdf_data[mask]), "SDF values do not match"
for idx in mismatch_indices:
    i, j, k = idx
    if abs(sdf_test[i, j, k] - sdf_data[i, j, k])*65 > 0.001:
        print(f"Index ({i}, {j}, {k}): sdf_test={sdf_test[i, j, k]*65}, sdf_data={sdf_data[i, j, k]*65}")

895
Index (14, 12, 22): sdf_test=2.56029536947608, sdf_data=2.559187173843384
Index (14, 12, 24): sdf_test=-0.9733204683288932, sdf_data=-0.9718682169914246
Index (15, 18, 24): sdf_test=-0.6311154272407293, sdf_data=-0.6297584772109985
Index (15, 19, 7): sdf_test=1.167785907164216, sdf_data=1.165723204612732
Index (15, 20, 7): sdf_test=2.019780734553933, sdf_data=2.01809024810791
Index (15, 20, 10): sdf_test=1.0601689387112856, sdf_data=1.0589762926101685
Index (15, 22, 20): sdf_test=0.7069492572918534, sdf_data=0.7059367299079895
Index (16, 5, 16): sdf_test=1.7331201676279306, sdf_data=1.7316709756851196
Index (16, 26, 18): sdf_test=2.0202510990202427, sdf_data=2.0166091918945312
Index (17, 19, 7): sdf_test=1.167786754667759, sdf_data=1.1657273769378662
Index (17, 20, 10): sdf_test=1.060180077329278, sdf_data=1.0589784383773804
Index (18, 22, 19): sdf_test=1.6494534071534872, sdf_data=1.6484432220458984


In [146]:
v, f,_, _ = measure.marching_cubes(sdf_data.copy(), level=0.0)
plot(v, f)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(16.0, 16.…

In [3]:
mask = sdf_32<0
mask.sum()

np.int64(422)